In [ ]:
import os
os.chdir('/Users/sebas/projects/metabolomics')
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
import os
import copy
import dill
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader
from rdkit import Chem
from simba.load_data import LoadData
from simba.config import Config
from simba.loader_saver import LoaderSaver
from simba.preprocessor import Preprocessor
from simba.transformers.encoder import Encoder
from simba.transformers.load_data_encoder import LoadDataEncoder
from simba.analog_discovery.analog_discovery import AnalogDiscovery
from simba.analog_discovery.mces import MCES
from simba.analog_discovery.fc_layers_analog_discovery import FcLayerAnalogDiscovery
from simba.tanimoto import Tanimoto
import simba
import sys
sys.modules['src'] = simba

## params

In [ ]:
MULTITASKING=True

In [ ]:
data_folder = '/Users/sebas/projects/data/'
janssen_path = data_folder + 'drug_plus.mgf'
output_janssen_file = data_folder + 'all_spectrums_janssen.pkl'
dataset_path = data_folder + 'edit_distance_neurips_nist_exhaustive.pkl'
#model_path = data_folder + 'review_models/best_model_contrastive_loss.ckpt'
model_path = data_folder + 'best_model_generated_data_20250130.ckpt'
casmi_file = data_folder + 'processed_massformer/spec_df.pkl'
msnlib_file = '/Users/sebas/projects/data/enamdisc_diverse_500.mgf'

In [ ]:
# Choose one query dataset. If both flags are False, use Janssen.
USE_CASMI = True
USE_MSNLIB = False
USE_MCES_RAW = False
USE_GNPS = False

if USE_CASMI and USE_MSNLIB:
    raise ValueError("Select only one query dataset: USE_CASMI or USE_MSNLIB")

query_dataset_name = 'casmi' if USE_CASMI else ('msnlib' if USE_MSNLIB else 'janssen')

In [ ]:
similarity_name = "simba_generated_data"

In [ ]:
if USE_GNPS:
    similarity_name=similarity_name + '_gnps'
# Include the query dataset in saved result filenames to avoid overwriting other runs.
similarity_name = f'{similarity_name}_{query_dataset_name}'

In [ ]:
config=Config()

In [ ]:
config.D_MODEL = 256
config.N_LAYERS = 5
config.TRANSFORMER_CONTEXT = 100
config.USE_FINGERPRINT = 0

In [ ]:
config.use_cosine_distance = True

## open janssen data

In [ ]:
if USE_CASMI:
    all_spectrums_janssen_original = LoadData.get_all_spectrums_casmi(
        casmi_file,
        config=config,
    )
elif USE_MSNLIB:
    from simba.simba.preprocessing_simba import PreprocessingSimba
    # Load MSnLib query spectra directly from the ENAMDISC MGF file.
    all_spectrums_janssen_original = PreprocessingSimba.load_spectra(
        msnlib_file, config
    )
else:
    loader_saver = LoaderSaver(
        block_size=100,
        pickle_nist_path='',
        pickle_gnps_path='',
        pickle_janssen_path=output_janssen_file,
    )
    all_spectrums_janssen_original = loader_saver.get_all_spectrums(
        janssen_path,
        100000000,
        use_tqdm=True,
        use_nist=False,
        config=config,
        use_janssen=True,
    )

In [ ]:
all_spectrums_janssen_processed= [copy.deepcopy(s) for s in all_spectrums_janssen_original]
pp=Preprocessor()
all_spectrums_janssen_processed = [pp.preprocess_spectrum(
            s,
            fragment_tol_mass=10,
            fragment_tol_mode="ppm",
            min_intensity=0.01,
            max_num_peaks=1000,
            scale_intensity=None,
        ) for s in all_spectrums_janssen_processed]


all_spectrums_janssen = [s_original for s_original, s_processed in zip(all_spectrums_janssen_original,all_spectrums_janssen_processed) if len(s_processed.mz)>=6]

## open ref data

In [ ]:
if USE_GNPS:
    from simba.simba.preprocessing_simba import PreprocessingSimba
    from matchms.importing import load_from_mgf,load_from_msp
    all_spectrums_reference_original = PreprocessingSimba.load_spectra(gnps_file, config, use_gnps_format=True)
    all_spectrums_reference_original = [s for s in all_spectrums_reference_original if Chem.MolFromSmiles(s.smiles) is not None]

    
else:
    with open(dataset_path, 'rb') as file:
            dataset = dill.load(file)
    all_spectrums_reference_original= dataset['molecule_pairs_train'].spectrums_original +\
                    dataset['molecule_pairs_val'].spectrums_original + \
                        dataset['molecule_pairs_test'].spectrums_original

In [ ]:
for i,s in enumerate(all_spectrums_reference_original):
    if 'pepmass' in s.params:
        all_spectrums_reference_original[i].params['pepmass']= s.params['pepmass'][0]

In [ ]:
for i,s in enumerate(all_spectrums_reference_original):
    if 'charge' in s.params:
        all_spectrums_reference_original[i].params['charge']= s.params['charge'][0]
    else:
        all_spectrums_reference_original[i].params['charge']=1

In [ ]:
all_spectrums_reference_processed= [copy.deepcopy(s) for s in all_spectrums_reference_original]
all_spectrums_reference_processed = [pp.preprocess_spectrum(
            s,
            fragment_tol_mass=10,
            fragment_tol_mode="ppm",
            min_intensity=0.01,
            max_num_peaks=1000,
            scale_intensity='root',
        ) for s in all_spectrums_reference_processed]
    
all_spectrums_reference = [s_original for s_original, s_processed in zip(all_spectrums_reference_original,all_spectrums_reference_processed) if len(s_processed.mz)>=6]

## Find those instances that are not in reference

In [ ]:
canon_smiles_reference = [Chem.CanonSmiles(s.smiles) for s in all_spectrums_reference ]
canon_smiles_janssen =   [Chem.CanonSmiles(s.smiles) for s in all_spectrums_janssen]
janssen_indexes_in_ref= [i for i,s in enumerate(canon_smiles_janssen) if s in canon_smiles_reference]
janssen_indexes_not_in_ref = [i for i,s in enumerate(canon_smiles_janssen) if s not in canon_smiles_reference]

In [ ]:
all_spectrums_janssen = [all_spectrums_janssen[index] for index in janssen_indexes_not_in_ref]

## load encoder

In [ ]:
encoder= Encoder(model_path, D_MODEL=int(config.D_MODEL),
                 N_LAYERS=int(config.N_LAYERS), 
                 multitasking=MULTITASKING, 
                 config=config)

## data loader

In [ ]:
dataset_janssen= LoadDataEncoder.from_spectrums_to_dataset(all_spectrums_janssen, max_num_peaks=int(config.TRANSFORMER_CONTEXT),)
dataloader_janssen= DataLoader(dataset_janssen, batch_size=config.BATCH_SIZE,  num_workers=0)

In [ ]:
dataset_reference= LoadDataEncoder.from_spectrums_to_dataset(all_spectrums_reference,  max_num_peaks=int(config.TRANSFORMER_CONTEXT),)
dataloader_reference= DataLoader(dataset_reference, batch_size=config.BATCH_SIZE,  num_workers=0)

## forward

In [ ]:
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
embeddings_janssen= encoder.get_embeddings(dataloader_janssen)

In [ ]:
embeddings_reference = encoder.get_embeddings(dataloader_reference)

## compute similarities simba

In [ ]:
similarities2, similarities1 = FcLayerAnalogDiscovery.compute_all_combinations(
    model_path, embeddings_janssen, embeddings_reference, config
)

In [ ]:
similarities2_norm = np.argmax(similarities2, axis=2)/5

In [ ]:
plt.hist(similarities1.reshape(-1)[0:10000])

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)
similarities2_softmax= softmax(similarities2)
similarities2_integer = similarities2_norm*5

In [ ]:
similarities1_integer = np.array((similarities1*40))
similarities1_integer= np.round(similarities1_integer)

In [ ]:
import numpy as np



def compute_ranking(
    similarities1,
    similarities1_integer,
    similarities2_integer,
    max_value_2_int=5
):
    '''
    rerank the results of the model by first (1) integer MCES distance (2) Edit distance and (3) float MCES distance
    If 2 pairs have the same integer MCES distance, use the edit distance for breaking the tie. If they tie yet use the floating MCES distance value.
    '''
    ranking_total = np.zeros(similarities1.shape, dtype=float)

    for row_index, (row_sim, row_int, row_int2) in enumerate(
        zip(similarities1, similarities1_integer, similarities2_integer)
    ):
        sorted_indices = np.lexsort(
            (-row_sim, -row_int2, -row_int)
        )

        ranking = np.empty_like(sorted_indices, dtype=float)
        ranking[sorted_indices] = np.arange(
            len(row_sim),
            0,
            -1
        )

        ranking /= len(row_sim)

        ranking_total[row_index] = ranking

    return ranking_total

In [ ]:
plt.hist(similarities2_softmax[:,:,-1].reshape(-1)[0:100000])

In [ ]:
ranking_total= compute_ranking(similarities1, similarities1_integer, similarities2_integer)

In [ ]:
similarities = ranking_total

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(similarities[:, 0:100], cmap='hot', interpolation='nearest')
plt.title('Heatmap of Matrix')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

## Check distribution of outputs

In [ ]:
N=1000
random_j= np.random.randint(0,similarities.shape[0],N)
random_r=np.random.randint(0,similarities.shape[1],N)

In [ ]:
plt.hist(similarities1[random_j,random_r], bins=20)

In [ ]:
plt.hist(similarities2_norm[random_j,random_r], bins=20)

In [ ]:
plt.hist(similarities[random_j,random_r], bins=20)

## Retrieval of best candidates

In [ ]:
spectrums_k_retrieved, tanimoto_k_retrieved, sim_k_retrieved, arg_max_k10 =\
                AnalogDiscovery.get_k_candidates(similarities, all_spectrums_reference, 
                                                 all_spectrums_janssen,k=10, return_arg_max_k10=True)

In [ ]:
if USE_MCES_RAW:
    mces_k_retrieved= []
    for s_group, spec_janssen in zip(spectrums_k_retrieved,all_spectrums_janssen):
        smiles_0 = [s.params['smiles'] for s in s_group]
        smiles_1 = [spec_janssen.params['smiles']]*len(s_group)
        df = MCES.compute_mces_list_smiles(smiles_0, smiles_1, threshold_mces=20)
        mces_k_retrieved.append(df['mces'].values)
else:
    mces_k_retrieved = [[MCES.calculate_mces_sim(s.params['smiles'], spec_janssen.params['smiles']) for s in s_group] for s_group, spec_janssen in zip(spectrums_k_retrieved,all_spectrums_janssen)]

In [ ]:
if USE_MCES_RAW:
    best_indexes = [np.argmin([m for m in mces_group if m is not None]) if len(mces_group)>=0 else 0 for mces_group in mces_k_retrieved ]
else: #if using normalized similarity
    best_indexes = [np.argmax([m for m in mces_group if m is not None]+[0]) if len(mces_group)>=0 else 0 for mces_group in mces_k_retrieved ]

In [ ]:
spectrums_retrieved = [spectrums_k_retrieved[index_spectrum][best_index]  for index_spectrum, best_index in enumerate(best_indexes) if best_index is not None]
tanimoto_retrieved = [tanimoto_k_retrieved[index_spectrum][best_index]  for index_spectrum, best_index in enumerate(best_indexes) if best_index is not None]
mces_sims = [mces_k_retrieved[index_spectrum][best_index]  for index_spectrum, best_index in enumerate(best_indexes) if best_index is not None]

max_sim = [sim_k_retrieved[index_spectrum][best_index]  for index_spectrum, best_index in enumerate(best_indexes) if best_index is not None]

In [ ]:
smiles_retrieved= [s.smiles for s in spectrums_retrieved]

In [ ]:
_=plt.hist(max_sim, color='r', bins=10)
plt.xlabel('maximum SIMBA predicted similarity found')
plt.ylabel('freq')
plt.grid()

In [ ]:
plt.hist(tanimoto_retrieved,density=True, bins=10)
plt.grid()
plt.xlabel('tanimoto similarity with reference spectra')
plt.ylabel('frequency')
plt.ylim([0,3.5])

In [ ]:
plt.boxplot(tanimoto_retrieved)
plt.grid()

In [ ]:
all_smiles_janssen = [s.params['smiles'] for s in all_spectrums_janssen]

In [ ]:
mces_sims=[]
for s0,s1 in tqdm(zip(all_smiles_janssen, smiles_retrieved)):
    similarity= MCES.calculate_mces_sim(s0, s1)
    mces_sims.append(similarity)

In [ ]:
plt.boxplot([m for m in mces_sims if m is not None])
plt.grid()

## Sanity check

In [ ]:
target_janssen_index= 0
target_ref_index=np.argsort(similarities[target_janssen_index])[::-1][0]

In [ ]:
spec_ref=    all_spectrums_reference[target_ref_index]
spec_janssen=all_spectrums_janssen[target_janssen_index]

In [ ]:
smiles_ref= spec_ref.params['smiles']
smiles_janssen= spec_janssen.params['smiles']

In [ ]:
mces_norm = similarities1[target_janssen_index, target_ref_index]
print(f'mces distance: {config.MCES20_MAX_VALUE*(1-mces_norm)}')

In [ ]:
ed_vector=similarities2[target_janssen_index, target_ref_index]
print(ed_vector)
print(f'Edit distance found: {5-np.argmax(ed_vector)} modifications')

## Save results

In [ ]:
results ={ 'smiles_janssen' : [s.params['smiles'] for s in all_spectrums_janssen], 
        'smiles_reference' : [s.params['smiles'] for s in all_spectrums_reference], 
        'mces_retrieved':mces_sims,
          'similarities':similarities,
          'similarities1':similarities1,
          'similarities2':similarities2,
         }

In [ ]:
results_dir = './notebooks_local/discovery_search/results'
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(
    results_dir,
    f'{similarity_name}_results_analog_discovery_unknown_compounds.pkl',
)
with open(results_file, 'wb') as f:
    dill.dump(results, f)
print(f'Saved {query_dataset_name.upper()} results to: {results_file}')

In [ ]:
target_index=58

In [ ]:
target_query=all_spectrums_janssen[target_index]

In [ ]:
smiles_query= target_query.params['smiles']

In [ ]:
# The ground-truth similarity matrix must correspond to the selected query dataset.
# Generate it separately if the dataset-specific file does not exist.
tanimoto_results_path = f'tanimoto_similarity_{query_dataset_name}_NEURIPS'
ground_truth_file = os.path.join(
    './notebooks_local/discovery_search/results',
    f'{tanimoto_results_path}_results_analog_discovery_unknwon_compounds.pkl',
)
if not os.path.exists(ground_truth_file):
    raise FileNotFoundError(
        f'No ground-truth matrix for {query_dataset_name.upper()} at {ground_truth_file}. '
        'Generate a dataset-matched Tanimoto similarity matrix before running the '
        'ground-truth comparison cells.'
    )
with open(ground_truth_file, 'rb') as f:
    golden_truth = dill.load(f)['similarities']
if golden_truth.shape != similarities.shape:
    raise ValueError(
        f'Ground-truth shape {golden_truth.shape} does not match prediction '
        f'shape {similarities.shape} for {query_dataset_name.upper()}.'
    )

In [ ]:
best_golden_truth_index=np.argsort(golden_truth[target_index])[::-1][0]

In [ ]:
max_value=np.sort(golden_truth[target_index])[-1]
best_golden_truth_list = np.where(golden_truth[target_index]==max_value)[0]